# 05 — Area Temporal GNN Training & Hierarchical Coordination
Trains the temporal area forecaster on ward traces, then runs a coordination check.
**Run 01, 02, 04 first.**

In [1]:
import sys, os
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path(os.getcwd()).resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')


Project root: D:\Bunker\BaseCamp\Hierarchical-multi-agent-RL-for-Urban-Traffic


## 1. Train Area GNN on collected ward data

In [2]:
import torch
from src.topology import Topology
from src.controllers.area_controller import AreaForecaster

topology = Topology(PROJECT_ROOT)

# --- CONFIGURATION: Select which Area to train the GNN on ---
# Options in your registry: 'HSR_Layout', 'BTM_Layout', 'BTM_Layout_1', 'Jayanagar'
TARGET_AREAS = ['HSR_Layout', 'BTM_Layout']

# Loading the consolidated temporal ward dataset
gnn_dir = PROJECT_ROOT / 'models' / 'gnn'
global_data_path = gnn_dir / 'global_temporal_data.pt'
if not global_data_path.exists():
    global_data_path = gnn_dir / 'global_gnn_data.pt'

if global_data_path.exists():
    all_data = torch.load(global_data_path, weights_only=False)
    print(f'⚡ Loaded {len(all_data)} temporal samples directly from {global_data_path.name}!')
    
    # Save the combined path for compatibility
    combined_path = gnn_dir / 'combined_training_data.pt'
    torch.save(all_data, combined_path)
    
    for area_id in TARGET_AREAS:
        forecaster = AreaForecaster(area_id, topology)
        print(f'\n🚀 Starting GPU-Powered Offline GNN Training on {forecaster.device} for {area_id}...')
        losses = forecaster.train_offline(combined_path, epochs=100, save_dir=gnn_dir)
        print(f'\n✅ GNN Training Complete for {area_id}! Final MSE Loss: {losses[-1]:.6f}')
else:
    print('❌ No global GNN data found. Please run the fast 20-episode collection run in notebook 04 first!')

⚡ Loaded 5999 temporal samples directly from global_temporal_data.pt!

🚀 Starting GPU-Powered Offline GNN Training on cuda for HSR_Layout...


KeyboardInterrupt: 

## 2. Run Full Hierarchical Simulation

In [ ]:
from src.runtime import run_simulation

result = run_simulation(
    scope='ward', identifier='ward_070',  # Test a ward from HSR Layout
    project_root=PROJECT_ROOT,
    gui=False,
    scenario_id='normal',
    max_ticks=1000,
    use_rl=True,
    use_area=True,
    use_city=True,
    collect_tick_records=False,
)

import json
print(json.dumps(result, indent=2, default=str))

## 3. Results

In [ ]:
results_dir = PROJECT_ROOT / 'results' / 'inference'
for f in sorted(results_dir.glob('*.json')):
    print(f'  {f.name}')

## 4. Full Sweep Evaluation Scorecard (Baseline vs Hierarchical AI Control)
Runs evaluations on representative wards and scenarios, comparing standard Baselines (No Control) with our complete Hierarchical AI Control stack (RL + GNN + MCMF).

In [ ]:
import pandas as pd
from src.evaluation.common.runner import run_case, EvaluationCase
from src.evaluation.common.metrics import normalize_run_metrics
from IPython.display import display, Markdown

# Select the wards representing HSR Layout and BTM Layout
eval_wards = ['ward_070', 'ward_071', 'ward_072', 'ward_017', 'ward_018']
eval_scenarios = ['peak_congestion', 'ambulance_emergency']
max_ticks = 1000  # stable timestep window for evaluation

results_table = []

print("🚀 Starting Sweep Evaluation: Comparing Baseline vs Hierarchical AI Control...")
print("--------------------------------------------------------------------------------")

for scenario in eval_scenarios:
    for ward in eval_wards:
        print(f"🔄 Evaluating {ward} | Scenario: {scenario}...")
        
        # 1. Run Baseline (No Control)
        case_base = EvaluationCase(
            scenario_id=scenario,
            ward_id=ward,
            max_ticks=max_ticks,
            gui=False,
            use_rl=False,
            use_area=False,
            use_city=False,
            algorithm='ppo'
        )
        res_base = run_case(PROJECT_ROOT, case_base, collect_tick_records=False)
        metrics_base = res_base["normalized_metrics"]
        
        # 2. Run Hierarchical AI Control (RL + GNN + City MCMF)
        case_ai = EvaluationCase(
            scenario_id=scenario,
            ward_id=ward,
            max_ticks=max_ticks,
            gui=False,
            use_rl=True,
            use_area=True,
            use_city=True,
            algorithm='ppo'
        )
        res_ai = run_case(PROJECT_ROOT, case_ai, collect_tick_records=False)
        metrics_ai = res_ai["normalized_metrics"]
        
        # Record results
        results_table.append({
            "Ward": ward,
            "Scenario": scenario,
            "Mode": "Baseline",
            "Avg Speed (m/s)": metrics_base["avg_speed"],
            "Congestion": metrics_base["congestion_score"],
            "Queue Length (m)": metrics_base["queue_length"],
            "Throughput": metrics_base["throughput"],
            "Wait Time (s)": metrics_base["waiting_time"],
            "Ambulance Delay (s)": metrics_base["ambulance_delay"]
        })
        results_table.append({
            "Ward": ward,
            "Scenario": scenario,
            "Mode": "Hierarchical AI",
            "Avg Speed (m/s)": metrics_ai["avg_speed"],
            "Congestion": metrics_ai["congestion_score"],
            "Queue Length (m)": metrics_ai["queue_length"],
            "Throughput": metrics_ai["throughput"],
            "Wait Time (s)": metrics_ai["waiting_time"],
            "Ambulance Delay (s)": metrics_ai["ambulance_delay"]
        })

# Compile into DataFrame
df = pd.DataFrame(results_table)

# Build overall summary averages
overall_summary = []
for mode in ["Baseline", "Hierarchical AI"]:
    mode_df = df[df["Mode"] == mode]
    overall_summary.append({
        "Mode": mode,
        "Avg Speed (m/s)": mode_df["Avg Speed (m/s)"].mean(),
        "Congestion": mode_df["Congestion"].mean(),
        "Queue Length (m)": mode_df["Queue Length (m)"].mean(),
        "Throughput (Total)": mode_df["Throughput"].sum(),
        "Wait Time (s)": mode_df["Wait Time (s)"].mean(),
        "Ambulance Delay (s)": mode_df["Ambulance Delay (s)"].mean()
    })
df_summary = pd.DataFrame(overall_summary)

print("\n================================================================================")
print("🏆 MASTER EVALUATION SCORECARD COMPLETE!")
print("================================================================================")

display(Markdown("### 📊 Overall Performance Summary"))
display(df_summary)

display(Markdown("### 🗺️ Detailed Ward-Level Sweep Metrics"))
display(df)